# 02 - Text Preprocessing

This notebook applies and evaluates the text preprocessing pipeline:

- Load both the SMS and email datasets
- Clean messages with `clean_text()` from `src/preprocessing.py`
- See how cleaning changes message lengths
- Vectorize with TF-IDF
- Save the processed data

> **Prerequisite:** datasets exist at `data/raw/sms_spam.csv` and `data/raw/email_spam.csv`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Allow importing the project's src modules regardless of where Jupyter was launched
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import (  # noqa: E402
    clean_text,
    load_email_data,
    load_sms_data,
    preprocess_data,
    save_processed_data,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load the datasets

In [ ]:
sms = load_sms_data()
email = load_email_data()
sms["source"] = "sms"
email["source"] = "email"

df = pd.concat([sms, email], ignore_index=True)
print(f"Total messages: {len(df):,}")
df["label"].value_counts()

## 2. Cleaning in action

`clean_text()` lowercases, strips URLs/numbers/punctuation, and collapses whitespace.

In [ ]:
sample = df.sample(5, random_state=42)
sample["cleaned"] = sample["message"].apply(clean_text)
sample[["message", "cleaned"]]

In [ ]:
df["char_len_raw"] = df["message"].str.len()
df["cleaned"] = df["message"].apply(clean_text)
df["char_len_clean"] = df["cleaned"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(df["char_len_raw"], bins=60, color="#1565c0")
axes[0].set_title("Raw message length (chars)")
axes[1].hist(df["char_len_clean"], bins=60, color="#00838f")
axes[1].set_title("Cleaned message length (chars)")
for ax in axes:
    ax.set_xlabel("Length")
    ax.set_ylabel("Frequency")
    ax.set_xlim(0, 600)
plt.tight_layout()
plt.show()

## 3. Vectorization (TF-IDF)

Convert cleaned text into numeric features with TF-IDF, using the same
configuration as `src/train.py` (10,000 features, unigrams + bigrams).

In [ ]:
vectorizer = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["cleaned"])
print(f"TF-IDF matrix shape: {X.shape}")
print(f"Sparsity: {100 * (1 - X.nnz / (X.shape[0] * X.shape[1])):.2f}%")
print(f"Sample features: {list(vectorizer.get_feature_names_out()[:10])}")

## 4. Save the processed data

Drop the intermediate exploration columns and persist the cleaned dataset
for use in `03_model_training.ipynb`.

In [ ]:
processed = preprocess_data(df.drop(columns=["cleaned", "char_len_raw", "char_len_clean"]))
path = save_processed_data(processed, "processed_notebook02.csv")
print(f"Saved {len(processed):,} rows to {path}")
processed.head()